# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_14 — Information Coefficient and Feature Decay

### Research question

How can we measure whether alpha features have predictive information, whether that information decays across horizons, and when a feature should be placed under review or retired?

This notebook follows naturally from:

```text
03_09_volatility_surface_alpha_signals
03_10_statistical_arbitrage_pairs
03_11_factor_orthogonalization_and_eviction
03_13_cointegration_error_correction_model
```

The previous notebooks created candidate signals. This notebook builds the **signal health dashboard**.

It covers:

1. cross-sectional Information Coefficient;
2. Rank IC;
3. IC information ratio;
4. IC hit rate;
5. horizon decay curves;
6. rolling IC stability;
7. exponentially weighted IC monitoring;
8. feature half-life estimation;
9. feature turnover and persistence;
10. placebo / shuffled-target checks;
11. train/validation/test IC comparison;
12. regime-conditioned IC;
13. multiple-testing warnings;
14. feature health score;
15. review / retire / keep decisions;
16. saved reports and manifest.

The key idea:

> A signal is not just good or bad. It has strength, stability, decay, cost, redundancy, and regime dependence — all of which should be monitored.

## 1. Information Coefficient

For a cross-sectional signal $f_{i,t}$ and forward return $r_{i,t+h}$:

$$
IC_t(h)=corr_i(f_{i,t},r_{i,t+h})
$$

where the correlation is taken across assets $i$ at date $t$.

A positive IC means assets with higher signal scores tend to have higher future returns.

### Rank IC

Rank IC uses ranks:

$$
RankIC_t(h)=corr_i(rank(f_{i,t}),rank(r_{i,t+h}))
$$

Rank IC is more robust to outliers and nonlinear monotonic relationships.

### IC Information Ratio

$$
ICIR = \frac{\mathbb E[IC_t]}{\operatorname{std}(IC_t)}\sqrt{252}
$$

This measures consistency, not just average strength.

## 2. Feature decay

A feature may forecast near-term returns but lose predictive power at longer horizons.

Define IC by horizon:

$$
IC(h) = \mathbb E_t[ corr_i(f_{i,t},r_{i,t+h}) ]
$$

A feature decay curve might look like:

- strong at 1 day;
- weaker at 5 days;
- gone by 21 days.

This is useful for:

1. choosing holding period;
2. estimating turnover;
3. matching signal horizon with execution horizon;
4. deciding whether a feature is stale.

## 3. Feature health

A practical feature health dashboard should report:

1. mean IC;
2. Rank IC;
3. ICIR;
4. hit rate;
5. horizon decay;
6. rolling IC trend;
7. latest IC versus historical IC;
8. turnover;
9. regime stability;
10. placebo comparison;
11. test-set degradation from train;
12. multiple-testing caution.

A signal should be reviewed or retired if:

- test IC is weak;
- rolling IC has decayed;
- turnover is too high;
- performance is regime-specific;
- it fails placebo controls;
- it is redundant with existing signals.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class ICDecayConfig:
    n_dates: int = 1_000
    n_assets: int = 150
    n_sectors: int = 10
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    annualisation: int = 252
    horizons: tuple = (1, 2, 5, 10, 21, 42)
    rolling_ic_window: int = 126
    ewma_lambda: float = 0.94
    turnover_cost_per_unit: float = 0.0005
    min_test_rank_ic: float = 0.01
    min_latest_rolling_ic: float = 0.005
    max_turnover: float = 0.70
    seed: int = 42


config = ICDecayConfig()
config

## 5. Synthetic feature panel

We simulate a cross-sectional panel with multiple feature types:

| Feature | Behaviour |
|---|---|
| `stable_momentum` | persistent positive IC |
| `fast_reversal` | short-horizon IC that decays quickly |
| `slow_value` | weaker but longer-horizon IC |
| `decaying_alpha` | strong early, weak later |
| `regime_alpha` | works only in calm regimes |
| `crowded_alpha` | starts good, later reverses |
| `noisy_feature` | no true signal |
| `redundant_clone` | duplicate of stable momentum |
| `high_turnover_signal` | predictive but expensive |
| `unique_new_signal` | moderate independent signal |

This lets the notebook identify different feature health states.

In [ ]:
def simulate_feature_panel(config: ICDecayConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2017-01-01", periods=config.n_dates)
    assets = [f"asset_{i:03d}" for i in range(config.n_assets)]
    sectors = rng.integers(0, config.n_sectors, size=config.n_assets)

    # Persistent latent asset characteristics.
    latent_momentum = rng.normal(0, 1, config.n_assets)
    latent_value = rng.normal(0, 1, config.n_assets)
    latent_quality = rng.normal(0, 1, config.n_assets)
    latent_unique = rng.normal(0, 1, config.n_assets)

    rows = []

    for t, date in enumerate(dates):
        regime_stress = 1 if (np.sin(2 * np.pi * t / 250) + 0.5 * rng.normal()) > 0.9 else 0
        market_shock = rng.standard_t(df=7) * 0.008
        sector_shocks = rng.normal(0, 0.004, config.n_sectors)

        # Time-varying premia.
        stable_premium = 0.0016
        reversal_premium = 0.0022
        value_premium = 0.0009
        decaying_premium = 0.0020 * max(0.0, 1.0 - t / (0.65 * config.n_dates))
        regime_premium = 0.0018 if regime_stress == 0 else -0.0005
        crowded_premium = 0.0017 if t < int(0.55 * config.n_dates) else -0.0008
        high_turnover_premium = 0.0014
        unique_premium = 0.0012

        for i, asset in enumerate(assets):
            sector = sectors[i]

            # Slowly evolving features.
            stable_momentum = 0.85 * latent_momentum[i] + 0.35 * rng.normal()
            slow_value = 0.90 * latent_value[i] + 0.45 * rng.normal()
            quality = 0.90 * latent_quality[i] + 0.35 * rng.normal()
            unique_new_signal = 0.85 * latent_unique[i] + 0.45 * rng.normal()

            # Fast signals.
            fast_reversal = rng.normal()
            high_turnover_signal = rng.normal()

            # Redundant and problematic features.
            redundant_clone = 0.92 * stable_momentum + 0.20 * rng.normal()
            decaying_alpha = 0.80 * latent_value[i] + 0.50 * rng.normal()
            regime_alpha = stable_momentum + 0.50 * rng.normal()
            crowded_alpha = 0.65 * latent_momentum[i] + 0.50 * rng.normal()
            noisy_feature = rng.normal()

            # Controls.
            size = rng.normal(0, 1) + 0.50 * latent_quality[i]
            beta = rng.normal(1, 0.20) + 0.10 * sector / max(config.n_sectors - 1, 1)
            volatility = np.exp(rng.normal(-4.5, 0.35)) * (1.5 if regime_stress else 1.0)

            # Forward returns by horizon are generated later by adding horizon-specific decay.
            base_noise = volatility * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)

            one_day_return = (
                0.00005
                + stable_premium * stable_momentum
                - reversal_premium * fast_reversal
                + value_premium * slow_value
                + decaying_premium * decaying_alpha
                + regime_premium * regime_alpha
                + crowded_premium * crowded_alpha
                + high_turnover_premium * high_turnover_signal
                + unique_premium * unique_new_signal
                + 0.0004 * quality
                + 0.0007 * beta * market_shock / 0.008
                + sector_shocks[sector]
                + base_noise
            )

            rows.append({
                "date": date,
                "asset": asset,
                "sector": int(sector),
                "stress_regime": regime_stress,
                "size": size,
                "beta": beta,
                "volatility": volatility,
                "stable_momentum": stable_momentum,
                "fast_reversal": fast_reversal,
                "slow_value": slow_value,
                "quality": quality,
                "decaying_alpha": decaying_alpha,
                "regime_alpha": regime_alpha,
                "crowded_alpha": crowded_alpha,
                "noisy_feature": noisy_feature,
                "redundant_clone": redundant_clone,
                "high_turnover_signal": high_turnover_signal,
                "unique_new_signal": unique_new_signal,
                "forward_return_1d": one_day_return
            })

    panel = pd.DataFrame(rows)

    # Create multi-horizon forward returns using horizon-specific decays and noise.
    # In a real panel, these would be computed from actual future price paths.
    for h in config.horizons:
        decay_fast = np.exp(-h / 3)
        decay_medium = np.exp(-h / 14)
        decay_slow = np.exp(-h / 45)

        rng_h = np.random.default_rng(config.seed + h)

        panel[f"forward_return_{h}d"] = (
            h * 0.00003
            + decay_medium * 0.0012 * panel["stable_momentum"]
            - decay_fast * 0.0018 * panel["fast_reversal"]
            + decay_slow * 0.0010 * panel["slow_value"]
            + decay_medium * 0.0010 * panel["unique_new_signal"]
            + decay_medium * 0.0012 * panel["regime_alpha"] * (1 - 1.5 * panel["stress_regime"])
            + decay_slow * 0.0007 * panel["quality"]
            + rng_h.normal(0, np.sqrt(h) * panel["volatility"])
        )

    return panel


panel = simulate_feature_panel(config)

panel.head()

In [ ]:
feature_cols = [
    "stable_momentum",
    "fast_reversal",
    "slow_value",
    "quality",
    "decaying_alpha",
    "regime_alpha",
    "crowded_alpha",
    "noisy_feature",
    "redundant_clone",
    "high_turnover_signal",
    "unique_new_signal"
]

control_cols = ["size", "beta", "volatility"]

target_cols = [f"forward_return_{h}d" for h in config.horizons]

pd.Series({
    "n_rows": len(panel),
    "n_dates": panel["date"].nunique(),
    "n_assets": panel["asset"].nunique(),
    "n_features": len(feature_cols),
    "targets": ", ".join(target_cols)
})

## 6. Cross-sectional winsorisation and z-scoring

For each date and feature:

1. winsorise cross-sectionally;
2. z-score cross-sectionally.

This ensures factor values are comparable through time:

$$
z_{i,t}=\frac{x_{i,t}-\bar x_t}{s_t}
$$

In [ ]:
def cross_sectional_winsorize_zscore(df, cols, lower=0.01, upper=0.99):
    out = df.copy()

    for col in cols:
        z_col = f"z_{col}"
        z_parts = []

        for date, g in out.groupby("date"):
            x = g[col].copy()
            lo = x.quantile(lower)
            hi = x.quantile(upper)
            x = x.clip(lo, hi)
            std = x.std(ddof=1)
            z = (x - x.mean()) / (std if std > 0 else 1.0)
            z_parts.append(pd.Series(z.values, index=g.index))

        out[z_col] = pd.concat(z_parts).sort_index()

    return out


panel_z = cross_sectional_winsorize_zscore(panel, feature_cols + control_cols)

z_feature_cols = [f"z_{c}" for c in feature_cols]
z_control_cols = [f"z_{c}" for c in control_cols]

panel_z[["date", "asset"] + z_feature_cols[:5]].head()

## 7. Chronological split

We split by date into:

1. train;
2. validation;
3. test.

Feature decay and health decisions are evaluated on validation and test, not only train.

In [ ]:
unique_dates = np.array(sorted(panel_z["date"].unique()))
n_dates = len(unique_dates)

train_end = int(config.train_fraction * n_dates)
validation_end = int((config.train_fraction + config.validation_fraction) * n_dates)

train_dates = unique_dates[:train_end]
validation_dates = unique_dates[train_end:validation_end]
test_dates = unique_dates[validation_end:]

train = panel_z[panel_z["date"].isin(train_dates)].copy()
validation = panel_z[panel_z["date"].isin(validation_dates)].copy()
test = panel_z[panel_z["date"].isin(test_dates)].copy()

split_metadata = {
    "n_dates_total": int(n_dates),
    "n_dates_train": int(len(train_dates)),
    "n_dates_validation": int(len(validation_dates)),
    "n_dates_test": int(len(test_dates)),
    "train_start": str(pd.Timestamp(train_dates[0])),
    "train_end": str(pd.Timestamp(train_dates[-1])),
    "validation_start": str(pd.Timestamp(validation_dates[0])),
    "validation_end": str(pd.Timestamp(validation_dates[-1])),
    "test_start": str(pd.Timestamp(test_dates[0])),
    "test_end": str(pd.Timestamp(test_dates[-1])),
}

pd.Series(split_metadata)

## 8. Daily IC and Rank IC

For every date, feature, and horizon, compute:

$$
IC_t(h)=corr(f_{i,t},r_{i,t+h})
$$

and:

$$
RankIC_t(h)=corr(rank(f_{i,t}),rank(r_{i,t+h}))
$$

In [ ]:
def daily_ic_by_horizon(df, features, horizons):
    rows = []

    for date, g in df.groupby("date"):
        for feature in features:
            x = g[feature]

            for h in horizons:
                target = f"forward_return_{h}d"
                y = g[target]

                ic = x.corr(y)
                rank_ic = x.rank().corr(y.rank())

                rows.append({
                    "date": date,
                    "feature": feature,
                    "horizon": h,
                    "ic": ic,
                    "rank_ic": rank_ic
                })

    return pd.DataFrame(rows)


ic_train_daily = daily_ic_by_horizon(train, z_feature_cols, config.horizons)
ic_validation_daily = daily_ic_by_horizon(validation, z_feature_cols, config.horizons)
ic_test_daily = daily_ic_by_horizon(test, z_feature_cols, config.horizons)

ic_test_daily.head()

## 9. IC summary metrics

For each feature and horizon, summarise:

- mean IC;
- mean Rank IC;
- IC volatility;
- ICIR;
- Rank ICIR;
- hit rate;
- t-style statistic.

This is a standard first-pass feature health report.

In [ ]:
def summarize_ic(ic_daily):
    rows = []

    for (feature, horizon), g in ic_daily.groupby(["feature", "horizon"]):
        ic = g["ic"].dropna()
        rank_ic = g["rank_ic"].dropna()

        mean_ic = ic.mean()
        std_ic = ic.std(ddof=1)
        mean_rank_ic = rank_ic.mean()
        std_rank_ic = rank_ic.std(ddof=1)

        n = len(ic)

        rows.append({
            "feature": feature,
            "horizon": horizon,
            "n_dates": n,
            "mean_ic": mean_ic,
            "std_ic": std_ic,
            "ic_ir": mean_ic / std_ic * np.sqrt(config.annualisation) if std_ic > 0 else np.nan,
            "hit_rate_positive_ic": (ic > 0).mean(),
            "t_style_stat_ic": mean_ic / (std_ic / np.sqrt(n)) if std_ic > 0 and n > 1 else np.nan,
            "mean_rank_ic": mean_rank_ic,
            "std_rank_ic": std_rank_ic,
            "rank_ic_ir": mean_rank_ic / std_rank_ic * np.sqrt(config.annualisation) if std_rank_ic > 0 else np.nan,
            "hit_rate_positive_rank_ic": (rank_ic > 0).mean(),
            "t_style_stat_rank_ic": mean_rank_ic / (std_rank_ic / np.sqrt(n)) if std_rank_ic > 0 and n > 1 else np.nan
        })

    return pd.DataFrame(rows).sort_values(["horizon", "mean_rank_ic"], ascending=[True, False])


ic_train_summary = summarize_ic(ic_train_daily)
ic_validation_summary = summarize_ic(ic_validation_daily)
ic_test_summary = summarize_ic(ic_test_daily)

ic_test_summary.head(15)

In [ ]:
h1_test = ic_test_summary[ic_test_summary["horizon"] == 1].sort_values("mean_rank_ic")

plt.figure(figsize=(12, 6))
plt.barh(h1_test["feature"], h1_test["mean_rank_ic"])
plt.axvline(0, linestyle="--")
plt.title("Test Mean Rank IC, 1-Day Horizon")
plt.xlabel("Mean Rank IC")
plt.ylabel("Feature")
plt.show()

## 10. Horizon decay curves

A feature's predictive power often decays with horizon.

We plot:

$$
h \mapsto \overline{RankIC}(h)
$$

A short-lived feature should decay quickly.

A slower fundamental-style feature may persist longer.

In [ ]:
decay_table = ic_test_summary.pivot(index="horizon", columns="feature", values="mean_rank_ic").sort_index()

decay_table

In [ ]:
plt.figure(figsize=(12, 6))
for feature in ["z_stable_momentum", "z_fast_reversal", "z_slow_value", "z_decaying_alpha", "z_unique_new_signal", "z_noisy_feature"]:
    if feature in decay_table.columns:
        plt.plot(decay_table.index, decay_table[feature], marker="o", label=feature)
plt.axhline(0, linestyle="--")
plt.title("Test Horizon Decay: Mean Rank IC")
plt.xlabel("Forward horizon, days")
plt.ylabel("Mean Rank IC")
plt.legend()
plt.show()

## 11. Feature half-life from IC decay

We estimate an IC half-life.

For each feature, compare absolute IC at horizon $h$ to IC at the shortest horizon.

A crude decay half-life is the first horizon where:

$$
|IC(h)| \leq \frac{1}{2}|IC(1)|
$$

This is heuristic, but useful for matching signal horizon to holding period.

In [ ]:
def estimate_ic_half_life(decay_table):
    rows = []

    horizons = list(decay_table.index)

    for feature in decay_table.columns:
        vals = decay_table[feature].dropna()

        if len(vals) == 0:
            continue

        base_h = vals.index.min()
        base_ic = abs(vals.loc[base_h])

        if base_ic < 1e-8:
            half_life = np.nan
        else:
            half_life = np.nan
            for h in vals.index:
                if abs(vals.loc[h]) <= 0.5 * base_ic:
                    half_life = h
                    break

        rows.append({
            "feature": feature,
            "base_horizon": base_h,
            "base_abs_rank_ic": base_ic,
            "ic_half_life_horizon": half_life
        })

    return pd.DataFrame(rows).sort_values("base_abs_rank_ic", ascending=False)


ic_half_life = estimate_ic_half_life(decay_table)

ic_half_life

## 12. Rolling IC and feature decay

A feature can decay through calendar time.

We compute rolling mean Rank IC:

$$
\overline{RankIC}_{t,w} = \frac{1}{w}\sum_{j=t-w+1}^{t}RankIC_j
$$

This helps identify stale or regime-dependent signals.

In [ ]:
def rolling_ic(ic_daily, window):
    frames = []

    for (feature, horizon), g in ic_daily.groupby(["feature", "horizon"]):
        g = g.sort_values("date").copy()
        g["rolling_mean_ic"] = g["ic"].rolling(window, min_periods=max(20, window // 4)).mean()
        g["rolling_mean_rank_ic"] = g["rank_ic"].rolling(window, min_periods=max(20, window // 4)).mean()
        frames.append(g)

    return pd.concat(frames, ignore_index=True)


ic_all_daily = daily_ic_by_horizon(panel_z, z_feature_cols, config.horizons)
rolling_ic_all = rolling_ic(ic_all_daily, config.rolling_ic_window)

rolling_ic_all.head()

In [ ]:
plt.figure(figsize=(12, 6))
for feature in ["z_stable_momentum", "z_decaying_alpha", "z_crowded_alpha", "z_unique_new_signal", "z_noisy_feature"]:
    g = rolling_ic_all[(rolling_ic_all["feature"] == feature) & (rolling_ic_all["horizon"] == 1)]
    plt.plot(g["date"], g["rolling_mean_rank_ic"], label=feature)
plt.axhline(0, linestyle="--")
plt.title("Rolling Mean Rank IC, 1-Day Horizon")
plt.xlabel("Date")
plt.ylabel("Rolling Rank IC")
plt.legend()
plt.show()

## 13. EWMA IC monitor

Rolling windows can react slowly.

An exponentially weighted IC monitor is:

$$
EWIC_t = \lambda EWIC_{t-1}+(1-\lambda)IC_t
$$

with:

$$
0<\lambda<1
$$

This creates a smoother but responsive feature health monitor.

In [ ]:
def ewma_ic_monitor(ic_daily, lam):
    frames = []

    for (feature, horizon), g in ic_daily.groupby(["feature", "horizon"]):
        g = g.sort_values("date").copy()
        ew = []
        current = 0.0

        for val in g["rank_ic"].fillna(0.0):
            current = lam * current + (1 - lam) * val
            ew.append(current)

        g["ewma_rank_ic"] = ew
        frames.append(g)

    return pd.concat(frames, ignore_index=True)


ewma_ic = ewma_ic_monitor(ic_all_daily, config.ewma_lambda)

ewma_ic.tail()

In [ ]:
plt.figure(figsize=(12, 6))
for feature in ["z_stable_momentum", "z_decaying_alpha", "z_crowded_alpha", "z_unique_new_signal", "z_noisy_feature"]:
    g = ewma_ic[(ewma_ic["feature"] == feature) & (ewma_ic["horizon"] == 1)]
    plt.plot(g["date"], g["ewma_rank_ic"], label=feature)
plt.axhline(0, linestyle="--")
plt.title("EWMA Rank IC Monitor, 1-Day Horizon")
plt.xlabel("Date")
plt.ylabel("EWMA Rank IC")
plt.legend()
plt.show()

## 14. Train / validation / test degradation

A feature can look good in train and fail in test.

We compare mean Rank IC by split.

Large degradation is a warning sign.

In [ ]:
def split_ic_comparison(train_summary, validation_summary, test_summary, horizon=1):
    t = train_summary[train_summary["horizon"] == horizon][["feature", "mean_rank_ic", "rank_ic_ir"]].rename(columns={
        "mean_rank_ic": "train_rank_ic",
        "rank_ic_ir": "train_rank_ic_ir"
    })

    v = validation_summary[validation_summary["horizon"] == horizon][["feature", "mean_rank_ic", "rank_ic_ir"]].rename(columns={
        "mean_rank_ic": "validation_rank_ic",
        "rank_ic_ir": "validation_rank_ic_ir"
    })

    s = test_summary[test_summary["horizon"] == horizon][["feature", "mean_rank_ic", "rank_ic_ir"]].rename(columns={
        "mean_rank_ic": "test_rank_ic",
        "rank_ic_ir": "test_rank_ic_ir"
    })

    out = t.merge(v, on="feature").merge(s, on="feature")
    out["train_to_test_decay"] = out["test_rank_ic"] - out["train_rank_ic"]
    out["validation_to_test_decay"] = out["test_rank_ic"] - out["validation_rank_ic"]

    return out.sort_values("test_rank_ic", ascending=False)


split_comparison_h1 = split_ic_comparison(
    ic_train_summary,
    ic_validation_summary,
    ic_test_summary,
    horizon=1
)

split_comparison_h1

In [ ]:
plt.figure(figsize=(12, 6))
x = np.arange(len(split_comparison_h1))
plt.bar(x - 0.25, split_comparison_h1["train_rank_ic"], width=0.25, label="train")
plt.bar(x, split_comparison_h1["validation_rank_ic"], width=0.25, label="validation")
plt.bar(x + 0.25, split_comparison_h1["test_rank_ic"], width=0.25, label="test")
plt.axhline(0, linestyle="--")
plt.xticks(x, split_comparison_h1["feature"], rotation=90)
plt.title("Mean Rank IC by Split, 1-Day Horizon")
plt.ylabel("Mean Rank IC")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Regime-conditioned IC

A feature may work only in calm regimes or only in stress.

We compute IC by stress regime.

This is important because a feature with good average IC may fail exactly when risk is highest.

In [ ]:
def regime_ic_summary(df, features, horizons, regime_col="stress_regime"):
    rows = []

    for regime, g_regime in df.groupby(regime_col):
        ic_daily = daily_ic_by_horizon(g_regime, features, horizons)
        summary = summarize_ic(ic_daily)
        summary["regime"] = regime
        rows.append(summary)

    return pd.concat(rows, ignore_index=True)


regime_ic_test = regime_ic_summary(test, z_feature_cols, config.horizons)

regime_ic_h1 = regime_ic_test[regime_ic_test["horizon"] == 1][
    ["feature", "regime", "mean_rank_ic", "rank_ic_ir"]
].pivot(index="feature", columns="regime", values="mean_rank_ic")

regime_ic_h1.columns = [f"regime_{c}_rank_ic" for c in regime_ic_h1.columns]

regime_ic_h1

In [ ]:
regime_plot = regime_ic_h1.dropna().copy()

plt.figure(figsize=(12, 6))
x = np.arange(len(regime_plot))
plt.bar(x - 0.2, regime_plot.iloc[:, 0], width=0.4, label=regime_plot.columns[0])
if regime_plot.shape[1] > 1:
    plt.bar(x + 0.2, regime_plot.iloc[:, 1], width=0.4, label=regime_plot.columns[1])
plt.axhline(0, linestyle="--")
plt.xticks(x, regime_plot.index, rotation=90)
plt.title("Test Rank IC by Regime, 1-Day Horizon")
plt.ylabel("Mean Rank IC")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Feature turnover and persistence

A signal can have positive IC but be expensive to trade.

We approximate turnover using rank-weight changes.

For each date:

1. convert factor ranks into dollar-neutral weights;
2. compute average absolute weight change from previous date.

High-turnover signals need stronger IC to survive costs.

In [ ]:
def factor_rank_weights(g, factor):
    ranks = g[factor].rank(pct=True).to_numpy()
    raw = ranks - np.mean(ranks)
    denom = np.sum(np.abs(raw))
    if denom == 0:
        return raw
    return raw / denom


def factor_turnover(df, factors):
    rows = []

    for factor in factors:
        prev_weights = None

        for date, g in df.groupby("date"):
            g = g.sort_values("asset")
            weights = factor_rank_weights(g, factor)

            if prev_weights is not None:
                turnover = float(np.sum(np.abs(weights - prev_weights)))
                rows.append({
                    "date": date,
                    "feature": factor,
                    "turnover": turnover
                })

            prev_weights = weights

    return pd.DataFrame(rows)


turnover_daily = factor_turnover(test, z_feature_cols)

turnover_summary = (
    turnover_daily
    .groupby("feature")
    .agg(
        mean_turnover=("turnover", "mean"),
        median_turnover=("turnover", "median"),
        p90_turnover=("turnover", lambda x: x.quantile(0.90))
    )
    .reset_index()
    .sort_values("mean_turnover", ascending=False)
)

turnover_summary

## 17. Turnover-adjusted IC

A rough turnover-adjusted score is:

$$
IC_{net} = IC - c \cdot turnover
$$

This is not a true cost-adjusted backtest.

It is a quick feature triage metric.

High-turnover factors need higher IC to justify implementation.

In [ ]:
turnover_adjusted = (
    ic_test_summary[ic_test_summary["horizon"] == 1][["feature", "mean_rank_ic"]]
    .merge(turnover_summary[["feature", "mean_turnover"]], on="feature", how="left")
)

turnover_adjusted["turnover_adjusted_rank_ic"] = (
    turnover_adjusted["mean_rank_ic"]
    - config.turnover_cost_per_unit * turnover_adjusted["mean_turnover"]
)

turnover_adjusted = turnover_adjusted.sort_values("turnover_adjusted_rank_ic", ascending=False)

turnover_adjusted

## 18. Feature correlation and redundancy

If two features have highly correlated scores, one may be redundant.

We compute average cross-sectional feature correlation across dates.

This complements the IC analysis.

In [ ]:
def average_feature_corr(df, features):
    matrices = []

    for date, g in df.groupby("date"):
        matrices.append(g[features].corr().values)

    avg = np.nanmean(np.stack(matrices), axis=0)

    return pd.DataFrame(avg, index=features, columns=features)


avg_feature_corr_test = average_feature_corr(test, z_feature_cols)

plt.figure(figsize=(10, 8))
plt.imshow(avg_feature_corr_test.values, vmin=-1, vmax=1)
plt.colorbar(label="Average cross-sectional correlation")
plt.xticks(range(len(z_feature_cols)), z_feature_cols, rotation=90)
plt.yticks(range(len(z_feature_cols)), z_feature_cols)
plt.title("Average Feature Correlation, Test")
plt.tight_layout()
plt.show()

In [ ]:
redundancy_rows = []

for i, f1 in enumerate(z_feature_cols):
    for j, f2 in enumerate(z_feature_cols):
        if j <= i:
            continue
        corr = avg_feature_corr_test.loc[f1, f2]
        if abs(corr) > 0.80:
            redundancy_rows.append({
                "feature_1": f1,
                "feature_2": f2,
                "avg_corr_test": corr
            })

redundancy_report = pd.DataFrame(redundancy_rows).sort_values("avg_corr_test", ascending=False)

redundancy_report

## 19. Placebo / shuffled target check

A signal pipeline should not find strong IC when the target is randomly permuted.

We run a placebo test by shuffling forward returns within each date.

The expected IC should be near zero.

If placebo IC is similar to real IC, the signal is suspicious.

In [ ]:
def placebo_ic_test(df, features, horizon=1, seed=123):
    rng = np.random.default_rng(seed)
    target = f"forward_return_{horizon}d"

    placebo_df = df.copy()
    shuffled_parts = []

    for date, g in placebo_df.groupby("date"):
        shuffled = g[target].to_numpy().copy()
        rng.shuffle(shuffled)
        shuffled_parts.append(pd.Series(shuffled, index=g.index))

    placebo_df[f"{target}_placebo"] = pd.concat(shuffled_parts).sort_index()

    rows = []

    for date, g in placebo_df.groupby("date"):
        for feature in features:
            ic = g[feature].rank().corr(g[f"{target}_placebo"].rank())
            rows.append({
                "date": date,
                "feature": feature,
                "horizon": horizon,
                "rank_ic_placebo": ic
            })

    placebo_daily = pd.DataFrame(rows)

    placebo_summary = (
        placebo_daily
        .groupby("feature")
        .agg(
            mean_placebo_rank_ic=("rank_ic_placebo", "mean"),
            std_placebo_rank_ic=("rank_ic_placebo", "std"),
            placebo_hit_rate=("rank_ic_placebo", lambda x: (x > 0).mean())
        )
        .reset_index()
    )

    return placebo_daily, placebo_summary


placebo_daily, placebo_summary = placebo_ic_test(test, z_feature_cols, horizon=1, seed=config.seed + 999)

placebo_comparison = (
    ic_test_summary[ic_test_summary["horizon"] == 1][["feature", "mean_rank_ic"]]
    .merge(placebo_summary, on="feature", how="left")
)

placebo_comparison["real_minus_placebo"] = (
    placebo_comparison["mean_rank_ic"] - placebo_comparison["mean_placebo_rank_ic"]
)

placebo_comparison.sort_values("real_minus_placebo", ascending=False)

## 20. Multiple testing warning

If we test many features across many horizons, some will look good by chance.

Number of tests:

$$
N_{\text{tests}} = N_{\text{features}} \times N_{\text{horizons}}
$$

A simple conservative adjustment is Bonferroni:

$$
p_{\text{threshold}}=\frac{0.05}{N_{\text{tests}}}
$$

This notebook does not rely on formal p-values because IC time series can be autocorrelated and labels can overlap, but it reports the number of tests and warns against data mining.

In [ ]:
n_tests = len(z_feature_cols) * len(config.horizons)
bonferroni_threshold = 0.05 / n_tests

multiple_testing_report = pd.Series({
    "n_features": len(z_feature_cols),
    "n_horizons": len(config.horizons),
    "n_feature_horizon_tests": n_tests,
    "bonferroni_0_05_threshold": bonferroni_threshold,
    "warning": "Many feature/horizon tests increase false discovery risk."
})

multiple_testing_report

## 21. Feature health score

We combine diagnostics into a feature health table.

Inputs:

- test Rank IC;
- latest rolling Rank IC;
- EWMA Rank IC;
- turnover;
- train-to-test degradation;
- regime instability;
- placebo gap;
- redundancy.

This is a governance tool, not a universal formula.

In [ ]:
def latest_metric(df, feature_col, value_col, horizon=1):
    rows = []
    for feature, g in df[df["horizon"] == horizon].groupby("feature"):
        g = g.sort_values("date")
        vals = g[value_col].dropna()
        rows.append({
            "feature": feature,
            f"latest_{value_col}": vals.iloc[-1] if len(vals) else np.nan
        })
    return pd.DataFrame(rows)


latest_rolling = latest_metric(rolling_ic_all, "feature", "rolling_mean_rank_ic", horizon=1)
latest_ewma = latest_metric(ewma_ic, "feature", "ewma_rank_ic", horizon=1)

regime_instability = regime_ic_h1.copy()
if regime_instability.shape[1] >= 2:
    regime_instability["regime_ic_spread"] = regime_instability.max(axis=1) - regime_instability.min(axis=1)
else:
    regime_instability["regime_ic_spread"] = 0.0

regime_instability = regime_instability[["regime_ic_spread"]].reset_index().rename(columns={"index": "feature"})

max_corr_to_other = []
for f in avg_feature_corr_test.index:
    vals = avg_feature_corr_test.loc[f].drop(f)
    max_corr_to_other.append({
        "feature": f,
        "max_abs_corr_to_other": vals.abs().max()
    })
max_corr_to_other = pd.DataFrame(max_corr_to_other)

health = (
    split_comparison_h1[["feature", "train_rank_ic", "validation_rank_ic", "test_rank_ic", "train_to_test_decay"]]
    .merge(latest_rolling, on="feature", how="left")
    .merge(latest_ewma, on="feature", how="left")
    .merge(turnover_summary[["feature", "mean_turnover"]], on="feature", how="left")
    .merge(placebo_comparison[["feature", "mean_placebo_rank_ic", "real_minus_placebo"]], on="feature", how="left")
    .merge(regime_instability, on="feature", how="left")
    .merge(max_corr_to_other, on="feature", how="left")
)

# Scores where higher health is better.
health["ic_score"] = health["test_rank_ic"].rank(pct=True)
health["latest_score"] = health["latest_rolling_mean_rank_ic"].rank(pct=True)
health["ewma_score"] = health["latest_ewma_rank_ic"].rank(pct=True)
health["placebo_gap_score"] = health["real_minus_placebo"].rank(pct=True)

# Penalties.
health["turnover_penalty"] = health["mean_turnover"].rank(pct=True)
health["redundancy_penalty"] = health["max_abs_corr_to_other"].rank(pct=True)
health["regime_instability_penalty"] = health["regime_ic_spread"].rank(pct=True)

health["feature_health_score"] = (
    0.30 * health["ic_score"]
    + 0.20 * health["latest_score"]
    + 0.15 * health["ewma_score"]
    + 0.15 * health["placebo_gap_score"]
    - 0.10 * health["turnover_penalty"]
    - 0.05 * health["redundancy_penalty"]
    - 0.05 * health["regime_instability_penalty"]
)

health = health.sort_values("feature_health_score", ascending=False)

health

## 22. Keep / review / retire decisions

A simple rule set:

### Keep

- test Rank IC is positive;
- latest rolling IC is positive;
- turnover is acceptable;
- real IC exceeds placebo;
- not highly redundant.

### Review

- weak IC but not clearly dead;
- regime-dependent;
- high turnover;
- unstable rolling IC.

### Retire

- weak or negative test IC;
- latest rolling IC below threshold;
- placebo gap small;
- high redundancy;
- very high turnover.

These are governance labels, not mechanical truth.

In [ ]:
def classify_feature_health(row, config):
    if (
        row["test_rank_ic"] >= config.min_test_rank_ic
        and row["latest_rolling_mean_rank_ic"] >= config.min_latest_rolling_ic
        and row["mean_turnover"] <= config.max_turnover
        and row["real_minus_placebo"] > 0
        and row["max_abs_corr_to_other"] < 0.90
    ):
        return "keep"

    if (
        row["test_rank_ic"] < 0
        or row["latest_rolling_mean_rank_ic"] < -0.005
        or row["real_minus_placebo"] <= 0
        or row["mean_turnover"] > 1.25
    ):
        return "retire"

    return "review"


health["decision"] = health.apply(lambda row: classify_feature_health(row, config), axis=1)

health[[
    "feature",
    "test_rank_ic",
    "latest_rolling_mean_rank_ic",
    "latest_ewma_rank_ic",
    "mean_turnover",
    "max_abs_corr_to_other",
    "real_minus_placebo",
    "feature_health_score",
    "decision"
]]

In [ ]:
plt.figure(figsize=(12, 6))
plot_health = health.sort_values("feature_health_score")
plt.barh(plot_health["feature"], plot_health["feature_health_score"])
plt.title("Feature Health Score")
plt.xlabel("Health score")
plt.ylabel("Feature")
plt.show()

plt.figure(figsize=(8, 5))
health["decision"].value_counts().plot(kind="bar")
plt.title("Feature Decisions")
plt.xlabel("Decision")
plt.ylabel("Count")
plt.show()

## 23. Toy long-short factor backtest

IC is not a full backtest.

We build a simple cross-sectional long-short diagnostic:

1. long top quintile;
2. short bottom quintile;
3. equal weight;
4. subtract turnover cost.

This is not a production backtest, but it checks whether IC translates into a basic portfolio diagnostic.

In [ ]:
def factor_rank_weights(g, feature, q=0.20):
    g = g.sort_values("asset")
    n = len(g)
    n_leg = max(1, int(q * n))

    ranks = g[feature].rank(method="first")
    weights = np.zeros(n)

    long_idx = ranks.nlargest(n_leg).index
    short_idx = ranks.nsmallest(n_leg).index
    idx_to_pos = {idx: pos for pos, idx in enumerate(g.index)}

    for idx in long_idx:
        weights[idx_to_pos[idx]] = 1.0 / n_leg

    for idx in short_idx:
        weights[idx_to_pos[idx]] = -1.0 / n_leg

    return weights, g


def long_short_factor_backtest(df, feature, target="forward_return_1d", cost_per_unit=0.0005):
    rows = []
    prev_weights = None

    for date, g in df.groupby("date"):
        weights, g_sorted = factor_rank_weights(g, feature)

        gross = float(weights @ g_sorted[target].to_numpy())

        if prev_weights is None:
            turnover = float(np.sum(np.abs(weights)))
        else:
            turnover = float(np.sum(np.abs(weights - prev_weights)))

        cost = cost_per_unit * turnover
        net = gross - cost

        rows.append({
            "date": date,
            "feature": feature,
            "gross_return": gross,
            "turnover": turnover,
            "cost": cost,
            "net_return": net
        })

        prev_weights = weights

    out = pd.DataFrame(rows)
    out["cum_net_return"] = (1 + out["net_return"]).cumprod()
    return out


bt_frames = []

for feature in health["feature"].tolist():
    bt_frames.append(long_short_factor_backtest(test, feature, cost_per_unit=config.turnover_cost_per_unit))

factor_bt = pd.concat(bt_frames, ignore_index=True)

def summarize_factor_bt(bt):
    rows = []

    for feature, g in bt.groupby("feature"):
        r = g["net_return"]

        rows.append({
            "feature": feature,
            "n_dates": len(g),
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "information_ratio": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "mean_turnover": float(g["turnover"].mean()),
            "total_cost": float(g["cost"].sum()),
            "total_net_return": float((1 + r).prod() - 1)
        })

    return pd.DataFrame(rows).sort_values("information_ratio", ascending=False)


factor_bt_summary = summarize_factor_bt(factor_bt)

factor_bt_summary

In [ ]:
plt.figure(figsize=(12, 6))
for feature in ["z_stable_momentum", "z_fast_reversal", "z_decaying_alpha", "z_unique_new_signal", "z_noisy_feature"]:
    g = factor_bt[factor_bt["feature"] == feature]
    if len(g):
        plt.plot(g["date"], g["cum_net_return"], label=feature)
plt.title("Toy Long-Short Factor Backtests")
plt.xlabel("Date")
plt.ylabel("Cumulative net return")
plt.legend()
plt.show()

## 24. Decision dashboard

We combine:

- IC health;
- turnover;
- placebo;
- long-short diagnostic.

This is the output a research team could review weekly or monthly.

In [ ]:
decision_dashboard = (
    health
    .merge(
        factor_bt_summary[["feature", "information_ratio", "annualised_return", "mean_turnover", "total_net_return"]].rename(columns={
            "mean_turnover": "backtest_mean_turnover"
        }),
        on="feature",
        how="left"
    )
    .sort_values(["decision", "feature_health_score"], ascending=[True, False])
)

decision_dashboard

## 25. Practical checklist for IC and decay analysis

Before trusting a feature:

1. **Cross-sectional IC**  
   Is the feature predictive across assets?

2. **Rank IC**  
   Does monotonic rank information exist?

3. **Horizon decay**  
   What holding period matches the signal?

4. **Rolling IC**  
   Is performance stable through time?

5. **EWMA monitor**  
   Is the latest signal health deteriorating?

6. **Train/validation/test comparison**  
   Did performance survive out of sample?

7. **Regime diagnostics**  
   Does it fail in stress?

8. **Placebo test**  
   Is real IC meaningfully above shuffled-target IC?

9. **Turnover**  
   Is the signal implementable after costs?

10. **Redundancy**  
   Is the feature just another version of an existing factor?

11. **Multiple testing**  
   Did we test too many features?

12. **Economic rationale**  
   Is there a plausible reason the feature should work?

## 26. Saving outputs

The notebook saves:

1. synthetic feature panel;
2. standardised panel;
3. daily IC tables;
4. IC summaries by split;
5. horizon decay table;
6. IC half-life estimates;
7. rolling IC table;
8. EWMA IC monitor;
9. split comparison;
10. regime IC report;
11. turnover report;
12. redundancy report;
13. placebo comparison;
14. multiple-testing report;
15. health dashboard;
16. toy factor backtests;
17. decision dashboard;
18. manifest.

In [ ]:
output_dir = Path("data/processed/information_coefficient_and_feature_decay_v1")
output_dir.mkdir(parents=True, exist_ok=True)

panel_path = output_dir / "synthetic_feature_panel.csv"
panel_z_path = output_dir / "standardized_feature_panel.csv"
ic_train_daily_path = output_dir / "ic_train_daily.csv"
ic_validation_daily_path = output_dir / "ic_validation_daily.csv"
ic_test_daily_path = output_dir / "ic_test_daily.csv"
ic_train_summary_path = output_dir / "ic_train_summary.csv"
ic_validation_summary_path = output_dir / "ic_validation_summary.csv"
ic_test_summary_path = output_dir / "ic_test_summary.csv"
decay_table_path = output_dir / "horizon_decay_table.csv"
ic_half_life_path = output_dir / "ic_half_life.csv"
rolling_ic_path = output_dir / "rolling_ic.csv"
ewma_ic_path = output_dir / "ewma_ic.csv"
split_comparison_path = output_dir / "split_ic_comparison_h1.csv"
regime_ic_path = output_dir / "regime_ic_test.csv"
turnover_daily_path = output_dir / "turnover_daily.csv"
turnover_summary_path = output_dir / "turnover_summary.csv"
feature_corr_path = output_dir / "feature_correlation_test.csv"
redundancy_path = output_dir / "redundancy_report.csv"
placebo_daily_path = output_dir / "placebo_daily.csv"
placebo_comparison_path = output_dir / "placebo_comparison.csv"
multiple_testing_path = output_dir / "multiple_testing_report.csv"
health_path = output_dir / "feature_health_table.csv"
factor_bt_path = output_dir / "toy_factor_backtest.csv"
factor_bt_summary_path = output_dir / "toy_factor_backtest_summary.csv"
decision_dashboard_path = output_dir / "decision_dashboard.csv"
split_metadata_path = output_dir / "split_metadata.json"
manifest_path = output_dir / "manifest.json"

panel.to_csv(panel_path, index=False)
panel_z.to_csv(panel_z_path, index=False)
ic_train_daily.to_csv(ic_train_daily_path, index=False)
ic_validation_daily.to_csv(ic_validation_daily_path, index=False)
ic_test_daily.to_csv(ic_test_daily_path, index=False)
ic_train_summary.to_csv(ic_train_summary_path, index=False)
ic_validation_summary.to_csv(ic_validation_summary_path, index=False)
ic_test_summary.to_csv(ic_test_summary_path, index=False)
decay_table.to_csv(decay_table_path)
ic_half_life.to_csv(ic_half_life_path, index=False)
rolling_ic_all.to_csv(rolling_ic_path, index=False)
ewma_ic.to_csv(ewma_ic_path, index=False)
split_comparison_h1.to_csv(split_comparison_path, index=False)
regime_ic_test.to_csv(regime_ic_path, index=False)
turnover_daily.to_csv(turnover_daily_path, index=False)
turnover_summary.to_csv(turnover_summary_path, index=False)
avg_feature_corr_test.to_csv(feature_corr_path)
redundancy_report.to_csv(redundancy_path, index=False)
placebo_daily.to_csv(placebo_daily_path, index=False)
placebo_comparison.to_csv(placebo_comparison_path, index=False)
multiple_testing_report.to_frame("value").to_csv(multiple_testing_path)
health.to_csv(health_path, index=False)
factor_bt.to_csv(factor_bt_path, index=False)
factor_bt_summary.to_csv(factor_bt_summary_path, index=False)
decision_dashboard.to_csv(decision_dashboard_path, index=False)
Path(split_metadata_path).write_text(json.dumps(split_metadata, indent=2, default=str))

manifest = {
    "dataset_name": "information_coefficient_and_feature_decay_outputs",
    "schema_version": "information_coefficient_and_feature_decay_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "feature_cols": feature_cols,
    "z_feature_cols": z_feature_cols,
    "target_cols": target_cols,
    "split_metadata": split_metadata,
    "multiple_testing_report": multiple_testing_report.to_dict(),
    "top_health_features": health.head(5).to_dict(orient="records"),
    "decisions": health["decision"].value_counts().to_dict(),
    "notes": [
        "Synthetic panel contains stable, fast-decay, slow, decaying, regime-dependent, crowded, noisy, redundant, high-turnover, and unique features.",
        "IC and Rank IC are computed cross-sectionally by date.",
        "Horizon decay, rolling IC, and EWMA IC are included for feature health monitoring.",
        "Placebo shuffled-target test is included as a sanity check.",
        "Feature health score combines IC, latest IC, EWMA IC, placebo gap, turnover, redundancy, and regime instability.",
        "Toy long-short backtest is diagnostic only and not a production portfolio simulation."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

panel_path, ic_test_summary_path, health_path, decision_dashboard_path, manifest_path

## 27. Limitations

### 27.1 Synthetic data

The notebook uses synthetic cross-sectional data.

Real factor research requires point-in-time data, survivorship-bias-free universes, corporate action handling, liquidity filters, and realistic trading assumptions.

### 27.2 IC is not PnL

IC measures cross-sectional association.

It does not fully model turnover, constraints, crowding, market impact, borrow, or portfolio risk.

### 27.3 Overlapping labels

Forward multi-day returns can overlap.

Formal statistical inference should adjust for overlap and autocorrelation.

### 27.4 Multiple testing

Testing many features and horizons increases false discovery risk.

This notebook reports the issue but does not perform a full false-discovery-rate procedure.

### 27.5 Simplified turnover

Turnover is based on rank-weight changes.

A real strategy needs portfolio optimisation and execution simulation.

### 27.6 Feature health score is subjective

The health score weights are governance choices.

A research team should calibrate thresholds to its own process.

### 27.7 Regime labels are simplified

Real regimes are not known cleanly and may need independent estimation.

## 28. Research frontier and extensions

Important extensions include:

1. **Newey-West IC inference**  
   Correct standard errors for autocorrelation and overlapping labels.

2. **False discovery rate control**  
   Adjust for many tested features.

3. **Bayesian feature decay**  
   Estimate posterior probability that a feature is still alive.

4. **Hierarchical factor clustering**  
   Detect redundant families of signals.

5. **Purged cross-validation**  
   Avoid leakage with overlapping forward returns.

6. **Cost-aware IC**  
   Penalise IC by turnover and liquidity.

7. **Regime-conditioned feature selection**  
   Keep features that work in specific regimes.

8. **Online feature monitoring**  
   Daily alerting for IC drawdown and decay.

9. **Feature store integration**  
   Save factor health metadata with production features.

10. **Chinese futures alpha library**  
   Monitor carry, momentum, term structure, night-session, volatility, and L1 microstructure signals.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_15_tree_models_for_return_prediction**  
   Combine features nonlinearly.

2. **03_16_feature_importance_and_shap_for_alpha**  
   Interpret predictive models.

3. **04_03_risk_model_factor_exposures**  
   Connect alpha factors to risk exposures.

4. **05_01_vectorized_backtest_engine**  
   Turn features into portfolio backtests.

5. **05_05_walk_forward_model_validation**  
   Proper live-like validation for signals.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use IC health monitoring in a full futures strategy.

## 30. Summary

This notebook implemented Information Coefficient and feature decay analysis.

It showed how to:

1. simulate a cross-sectional feature panel;
2. compute IC and Rank IC;
3. summarise ICIR and hit rate;
4. build horizon decay curves;
5. estimate IC half-life;
6. monitor rolling IC;
7. monitor EWMA IC;
8. compare train, validation, and test IC;
9. diagnose regime-dependent IC;
10. estimate feature turnover;
11. detect redundancy;
12. run placebo target checks;
13. report multiple-testing risk;
14. compute a feature health score;
15. classify features as keep, review, or retire;
16. run toy long-short factor diagnostics;
17. save dashboards and metadata.

The key computational takeaway:

> Feature research needs monitoring through time and across horizons, not just one average IC number.

The key financial takeaway:

> A signal should remain in the library only if it is distinct, stable, implementable, and still predictive out of sample.

## 31. Further reading

- Grinold and Kahn, *Active Portfolio Management.*
- Alphalens-style factor evaluation methodology.
- Harvey, Liu, and Zhu on multiple testing and the factor zoo.
- Bailey et al. on backtest overfitting.
- Literature on IC analysis, rank IC, feature decay, factor turnover, and signal governance.